# RegDocAI Day 4: Tesseract vs PaddleOCR

This notebook completes the controlled OCR comparison on the same official FDA form pages, exact field rectangles, public Moderna protocol metadata, controlled test identifiers, and deterministic scan degradations used by the repository. It does not download or substitute a different dataset.


## 1. Upload the RegDocAI Day 4 ZIP
Upload the repository ZIP supplied with this milestone. The notebook locates the extracted project root automatically.


In [ ]:
from google.colab import files
from pathlib import Path
import shutil, zipfile

uploaded = files.upload()
archive_name = next(name for name in uploaded if name.lower().endswith('.zip'))
extract_root = Path('/content/regdocai_day4')
shutil.rmtree(extract_root, ignore_errors=True)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(archive_name) as archive:
    archive.extractall(extract_root)
project_candidates = list(extract_root.rglob('pyproject.toml'))
assert project_candidates, 'No pyproject.toml found in uploaded archive'
PROJECT_ROOT = project_candidates[0].parent
print(PROJECT_ROOT)


## 2. Install Tesseract, the core package, and the optional PaddleOCR runtime


In [ ]:
import subprocess, sys
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-paddle.txt')], check=True)


## 3. Regenerate deterministic degraded images when omitted from the compact ZIP
The archive keeps the real source form pages, exact ground truth, and degradation manifest but may omit derived PNGs to reduce file size. This cell regenerates them with the fixed repository seed.


In [ ]:
image_root = PROJECT_ROOT / 'data/processed/degraded_forms/images'
png_count = len(list(image_root.rglob('*.png'))) if image_root.exists() else 0
if png_count < 64:
    subprocess.run([sys.executable, 'scripts/generate_degraded_forms.py'], cwd=PROJECT_ROOT, check=True)
print('Degraded PNG count:', len(list(image_root.rglob('*.png'))))


## 4. Verify the project and optional PaddleOCR runtime


In [ ]:
subprocess.run([sys.executable, 'scripts/check_paddleocr_runtime.py'], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT_ROOT, check=True)


## 5. Run both engines on the identical 160 field instances
`--strict` prevents an incomplete run from being mistaken for a completed comparison.


In [ ]:
subprocess.run([
    sys.executable, 'scripts/benchmark_ocr_engines.py',
    '--engines', 'tesseract', 'paddleocr', '--strict'
], cwd=PROJECT_ROOT, check=True)


## 6. Inspect the completed comparison


In [ ]:
import pandas as pd, json
result_root = PROJECT_ROOT / 'results/ocr_engine_benchmark'
display(pd.read_csv(result_root / 'summary_overall_completed_engines.csv'))
display(pd.read_csv(result_root / 'summary_by_condition_completed_engines.csv'))
print(json.dumps(json.loads((result_root / 'benchmark_status.json').read_text()), indent=2))


## 7. Download the measured comparison outputs


In [ ]:
output_zip = shutil.make_archive('/content/regdocai_ocr_engine_results', 'zip', result_root)
files.download(output_zip)
